# Code Based Grading

A model grader is good at *task following* but flaky on *format* and *syntax*. Those two are deterministic — so grade them with **code**, not a model.

| Criterion | Grader |
|-----------|--------|
| **Format** — return only Python / JSON / Regex, no explanation | code |
| **Valid syntax** — output actually parses | code |
| **Task following** — accurate, addresses the request | model |

This notebook: three syntax validators, a `grade_syntax` dispatcher keyed on each task's `format`, a tightened prompt-under-test, and a combined score = **average(model, syntax)**.

> **Adaptations:**
> - **Haiku 4.5** throughout — both the improved prompt-under-test and the model grader use assistant prefilling (400s on flagships).
> - `get_text()` helper; explicit dataset path (kernel cwd is the repo root).
> - We **regenerate `dataset.json`** so each task carries a `"format"` field — the code grader needs it to pick a validator. (Lesson 02's dataset only had `"task"`.)

## Setup — helpers, improved `run_prompt`, model grader

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import json
import ast
import re
from statistics import mean
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"   # prefill used in prompt-under-test AND grader


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return get_text(message)


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Regenerate the dataset — now with a `format` field

The code grader needs to know which validator to run for each task, so the dataset gains a `"format"` of `"python"`, `"json"`, or `"regex"`. We add it to the generator's example output and re-save `dataset.json`.

In [2]:
DATASET_PATH = "building-with-the-claude-api/02-prompt-evaluation/dataset.json"


def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


dataset = generate_dataset()
with open(DATASET_PATH, "w") as f:
    json.dump(dataset, f, indent=2)

dataset

[{'task': "Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
  'format': 'regex'},
 {'task': 'Create a JSON CloudFormation template for an IAM role that trusts the EC2 service',
  'format': 'json'},
 {'task': "Write a Python function that takes an AWS region code and returns the corresponding region name (e.g., 'us-east-1' returns 'N. Virginia')",
  'format': 'python'}]

## Improved prompt-under-test

Tighten the instructions to demand *only* code, no commentary, and prefill with a generic ` ```code ` fence so Claude jumps straight into raw output without us having to name the language up front. Stop on ` ``` ` so we get just the body.

In [3]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")   # prefill: start raw code, no language preamble
    output = chat(messages, stop_sequences=["```"])
    return output

## The model grader (from the last lesson)

Still handles *task following* — we blend its score with the syntax score below.

In [4]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## Syntax validators + dispatcher

Each validator tries to *parse* the output as its language: success → `10`, failure → `0`. `grade_syntax` picks the right one from the test case's `format`.

In [5]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(output, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(output)
    elif format == "python":
        return validate_python(output)
    else:
        return validate_regex(output)

## Combine the scores

`run_test_case` now blends both graders — equal weight, simple average. Adjust the weights if technical correctness matters more (or less) than content quality for your use case.

In [6]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result with both graders"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": model_grade["reasoning"],
    }


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

## Run it

Load the freshly-regenerated dataset (with `format`) and run the full two-grader eval. The number itself isn't "good" or "bad" — it's a **baseline**. The whole point is that you can now change the prompt and *measure* whether the score moves.

In [7]:
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average score: 8.333333333333334
[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket_name(s3_uri):\n    match = re.match(r's3://([^/]+)', s3_uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
      "format": "regex"
    },
    "score": 8.5,
    "reasoning": "The solution effectively solves the core task with a straightforward regex approach. However, it lacks robustness for production use. Adding input validation, a docstring, and optionally bucket name constraint validation would improve reliability. The current implementation is suitable for basic use cases but should be hardened before deployment in real AWS workflows."
  },
  {
    "output": "\n{\n  \"AWSTemplateFormatVersion\": \"2010-09-09\",\n  \"Resources\": {\n    \"EC2TrustRole\": {\n      \"Type\": \"AWS::IAM::Role\",\n      \"Properties\": {\n        \"AssumeRole

## Takeaway

We now have a full eval: dataset → prompt → **code grader** (format + syntax, deterministic) + **model grader** (task following, flexible), blended into one number. That number turns prompt engineering from vibes into measurement — which is exactly what the next section (**Prompt engineering techniques**) gives you levers to move.